# Master Pipeline Runner — All 18 Scenarios × 10 Runs

**Resume-safe.** Stop the kernel any time — when you re-run cell 4 it skips runs already in the CSV.

**What it produces:**
- `master_run_results/master_pipeline_runs.csv` — one row per run with latency + structural scores
- `master_run_results/configs/` — one JSON per run
- `master_run_results/renders/` — one PNG per successful render

**Cost:** ~$3–8 OpenAI charges total. **Time:** ~2-4 hours wall-clock.

## Cell 1 — Setup

In [1]:
# ═══════════════════════════════════════════════════════════════════
# CELL 1 — Setup. Resume-safe master pipeline runner.
# ═══════════════════════════════════════════════════════════════════
import sys, os, time, json, copy, csv, traceback, hashlib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

CANDIDATES = [
    "/Users/srujanakadambari/Desktop/ffresh thesis/data-to-visual/data-to-visual-nicos-branch",
    os.path.abspath(".."), os.getcwd(),
]
PROJECT_ROOT = next(
    (p for p in CANDIDATES if os.path.isfile(os.path.join(p, "generate_visualization.py"))), None
)
os.chdir(PROJECT_ROOT)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from retrieve_data import retrieve_data
from generate_visualization import Agent
from init_phoenix import init_phoenix
from visualization_from_template import generate_from_template

import phoenix as px
if not hasattr(px, "_thesis_initialised"):
    client, tool_client, tracer = init_phoenix("master-architecture-180")
    px._thesis_initialised = True
    px._client = client; px._tool = tool_client; px._tracer = tracer
else:
    client = px._client; tool_client = px._tool; tracer = px._tracer

MD_TABLE = retrieve_data(None, type="test")
QUESTION = ("Wieviel Umsatz hatte Teckentrup in den Jahren 2021 bis 2024 im Segment JVA? "
            "Provide a detailed analysis and a comprehensive visualization.")

ROOT_OUT = "master_run_results"
os.makedirs(f"{ROOT_OUT}/configs", exist_ok=True)
os.makedirs(f"{ROOT_OUT}/renders", exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Output dir:   {ROOT_OUT}/")


🔭 OpenTelemetry Tracing Details 🔭
|  Phoenix Project: master-architecture-180
|  Span Processor: SimpleSpanProcessor
|  Collector Endpoint: http://localhost:6006/v1/traces
|  Transport: HTTP + protobuf
|  Transport Headers: {}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  ⚠️ WARNING: It is strongly advised to use a BatchSpanProcessor in production environments.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.

Project root: /Users/srujanakadambari/Desktop/ffresh thesis/data-to-visual/data-to-visual-nicos-branch
Output dir:   master_run_results/


## Cell 2 — Define all 18 scenario runners

In [2]:
# ═══════════════════════════════════════════════════════════════════
# CELL 2 — CORRECTED: 18 scenario runners
# Uses Agent.generate_visualization() for full pipeline (S0, SA1)
# Uses justify_chart_type as a string-return that updates cfg["charttype"]
# ═══════════════════════════════════════════════════════════════════

def _apply_justify(agent, cfg, analysis):
    """justify_chart_type returns a STRING — apply it to cfg['charttype']."""
    if cfg is None: return None
    if hasattr(cfg, "model_dump"): cfg = cfg.model_dump()
    if not isinstance(cfg, dict): return cfg
    try:
        chart_type_str = agent.justify_chart_type(analysis, MD_TABLE)
        if chart_type_str:
            cfg["charttype"] = chart_type_str
    except Exception as e:
        print(f"  [justify warning: {e}]", end=" ")
    return cfg

def run_S0(question):
    """S0: 4-call all o4-mini baseline using full generate_visualization()."""
    a = Agent(client, tool_client, tracer, "o4-mini")
    return a.generate_visualization(MD_TABLE, question)

def run_S1(question):
    """S1: 3-call o4-mini, no router (analyze + extract + justify)."""
    a = Agent(client, tool_client, tracer, "o4-mini")
    analysis = a.analyze_data(MD_TABLE, question)
    cfg = a.extract_chart_config(MD_TABLE, analysis=analysis)
    return _apply_justify(a, cfg, analysis)

def run_S2(question):
    a = Agent(client, tool_client, tracer, "o4-mini")
    analysis = a.analyze_data(MD_TABLE, question)
    return a.extract_chart_config(MD_TABLE, analysis=analysis)

def run_S3(question):
    a = Agent(client, tool_client, tracer, "o4-mini")
    return a.extract_chart_config(MD_TABLE, analysis=question)

def run_S4(question):
    a_o4 = Agent(client, tool_client, tracer, "o4-mini")
    a_4m = Agent(client, tool_client, tracer, "gpt-4o-mini")
    analysis = a_o4.analyze_data(MD_TABLE, question)
    cfg = a_4m.extract_chart_config(MD_TABLE, analysis=analysis)
    return _apply_justify(a_4m, cfg, analysis)

def run_S4b(question):
    a_4o = Agent(client, tool_client, tracer, "gpt-4o")
    a_4m = Agent(client, tool_client, tracer, "gpt-4o-mini")
    analysis = a_4o.analyze_data(MD_TABLE, question)
    cfg = a_4m.extract_chart_config(MD_TABLE, analysis=analysis)
    return _apply_justify(a_4m, cfg, analysis)

def run_S5(question):
    a_o4 = Agent(client, tool_client, tracer, "o4-mini")
    a_4m = Agent(client, tool_client, tracer, "gpt-4o-mini")
    analysis = a_o4.analyze_data(MD_TABLE, question)
    return a_4m.extract_chart_config(MD_TABLE, analysis=analysis)

def run_S5b(question):
    a_4o = Agent(client, tool_client, tracer, "gpt-4o")
    a_4m = Agent(client, tool_client, tracer, "gpt-4o-mini")
    analysis = a_4o.analyze_data(MD_TABLE, question)
    return a_4m.extract_chart_config(MD_TABLE, analysis=analysis)

def run_S6(question):
    a = Agent(client, tool_client, tracer, "gpt-4o-mini")
    return a.extract_chart_config(MD_TABLE, analysis=question)

def run_S7(question):
    a_o3 = Agent(client, tool_client, tracer, "o3")
    a_4m = Agent(client, tool_client, tracer, "gpt-4o-mini")
    analysis = a_o3.analyze_data(MD_TABLE, question)
    cfg = a_4m.extract_chart_config(MD_TABLE, analysis=analysis)
    return _apply_justify(a_4m, cfg, analysis)

def run_S8(question):
    a_o3 = Agent(client, tool_client, tracer, "o3")
    a_4m = Agent(client, tool_client, tracer, "gpt-4o-mini")
    analysis = a_o3.analyze_data(MD_TABLE, question)
    return a_4m.extract_chart_config(MD_TABLE, analysis=analysis)

def run_S9(question):
    a = Agent(client, tool_client, tracer, "gpt-4o")
    return a.extract_chart_config(MD_TABLE, analysis=question)

def run_SA1(question):
    """SA1: full pipeline via generate_visualization() with o4-mini."""
    a = Agent(client, tool_client, tracer, "o4-mini")
    return a.generate_visualization(MD_TABLE, question)

def run_SA2(question):
    a_o4 = Agent(client, tool_client, tracer, "o4-mini")
    a_4m = Agent(client, tool_client, tracer, "gpt-4o-mini")
    analysis = a_o4.analyze_data(MD_TABLE, question)
    cfg = a_4m.extract_chart_config(MD_TABLE, analysis=analysis)
    return _apply_justify(a_4m, cfg, analysis)

def run_SA3(question):
    a_4o = Agent(client, tool_client, tracer, "gpt-4o")
    a_4m = Agent(client, tool_client, tracer, "gpt-4o-mini")
    analysis = a_4o.analyze_data(MD_TABLE, question)
    cfg = a_4m.extract_chart_config(MD_TABLE, analysis=analysis)
    return _apply_justify(a_4m, cfg, analysis)

def run_SA4(question):
    a_4o = Agent(client, tool_client, tracer, "gpt-4o")
    a_4m = Agent(client, tool_client, tracer, "gpt-4o-mini")
    analysis = a_4o.analyze_data(MD_TABLE, question)
    return a_4m.extract_chart_config(MD_TABLE, analysis=analysis)

def run_SA5(question):
    a = Agent(client, tool_client, tracer, "gpt-4o-mini")
    return a.extract_chart_config(MD_TABLE, analysis=question)

def run_SA6(question):
    a_4o = Agent(client, tool_client, tracer, "gpt-4o")
    a_4m = Agent(client, tool_client, tracer, "gpt-4o-mini")
    analysis = a_4o.analyze_data(MD_TABLE, question)
    return a_4m.extract_chart_config(MD_TABLE, analysis=analysis)


SCENARIO_RUNNERS = {
    "S0":  ("4-call: router+analyze+extract+justify [o4-mini]",         run_S0),
    "S1":  ("3-call: analyze+extract+justify [o4-mini, no router]",     run_S1),
    "S2":  ("2-call: analyze+extract [o4-mini]",                        run_S2),
    "S3":  ("1-call: single-shot extract [o4-mini]",                    run_S3),
    "S4":  ("3-call: analyze[o4-mini]+extract[4o-mini]+justify",        run_S4),
    "S4b": ("3-call: analyze[gpt-4o]+extract[4o-mini]+justify",         run_S4b),
    "S5":  ("2-call: analyze[o4-mini]+extract[4o-mini]",                run_S5),
    "S5b": ("2-call: analyze[gpt-4o]+extract[4o-mini]",                 run_S5b),
    "S6":  ("1-call: single-shot [gpt-4o-mini]",                        run_S6),
    "S7":  ("3-call: analyze[o3]+extract[4o-mini]+justify",             run_S7),
    "S8":  ("2-call: analyze[o3]+extract[4o-mini]",                     run_S8),
    "S9":  ("1-call: single-shot [gpt-4o]",                             run_S9),
    "SA1": ("4-call: router+analyze+extract+justify [o4-mini]",         run_SA1),
    "SA2": ("4-call: router+analyze[o4-mini]+extract+justify[4o-mini]", run_SA2),
    "SA3": ("4-call: router+analyze[gpt-4o]+extract+justify[4o-mini]",  run_SA3),
    "SA4": ("3-call: router+analyze[gpt-4o]+extract[4o-mini]",          run_SA4),
    "SA5": ("2-call: router+single-shot[4o-mini]",                      run_SA5),
    "SA6": ("3-call: router[gpt-4o]+analyze[gpt-4o]+extract[4o-mini]",  run_SA6),
}

print(f"Defined {len(SCENARIO_RUNNERS)} scenario runners — all using correct method calls")

Defined 18 scenario runners — all using correct method calls


## Cell 3 — Validator + render helper

In [3]:
# ═══════════════════════════════════════════════════════════════════
# CELL 3 — Quality validator + render helper
# ═══════════════════════════════════════════════════════════════════

VALID_CT = {"line","bar","pie","scatter","column","stackedbar"}

def normalize_config(raw):
    """Convert anything (Pydantic model, dict-with-content, string) to a dict."""
    if raw is None: return None
    if hasattr(raw, "model_dump"): raw = raw.model_dump()
    if isinstance(raw, dict) and "content" in raw and isinstance(raw["content"], str):
        try:
            raw = json.loads(raw["content"])
        except Exception:
            pass
    if not isinstance(raw, dict):
        try: raw = dict(raw)
        except Exception: return None
    return raw


def validate(cfg, render=True, render_path=None):
    """8-check structural validator. Optionally render to PNG."""
    r = {"schema_complete":False,"valid_charttype":False,"has_data":False,
         "has_title":False,"has_axes":False,"renderable":False,
         "has_annotations":False,"data_consistency":False,
         "quality_score":0,"passed":False,"error":None,
         "n_data_groups":0,"n_annotations":0,"title":"","charttype":""}

    if cfg is None: r["error"]="null"; return r
    if not isinstance(cfg, dict): r["error"]="not dict"; return r

    req = ["titlename","charttype","xlabel","ylabel","data"]
    r["schema_complete"] = all(cfg.get(f) for f in req)

    ct = cfg.get("charttype","")
    if hasattr(ct,"value"): ct = ct.value
    r["charttype"] = str(ct).lower().strip()
    r["valid_charttype"] = r["charttype"] in VALID_CT

    data = cfg.get("data") or []
    r["has_data"] = len(data) > 0
    r["n_data_groups"] = len(data)
    r["has_title"] = bool(str(cfg.get("titlename","")).strip())
    r["title"] = str(cfg.get("titlename",""))[:80]
    r["has_axes"] = bool(cfg.get("xlabel")) and bool(cfg.get("ylabel"))

    consistent = True
    for g in data:
        d = g if isinstance(g,dict) else (g.model_dump() if hasattr(g,"model_dump") else {})
        xd, yd = d.get("x_data",[]) or [], d.get("y_data",[]) or []
        if not xd or not yd or len(xd) != len(yd): consistent=False; break
    r["data_consistency"] = consistent

    annos = cfg.get("annotations") or []
    r["has_annotations"] = len(annos) > 0
    r["n_annotations"] = len(annos)

    if render and r["has_data"] and r["valid_charttype"]:
        try:
            plt.close("all")
            generate_from_template(copy.deepcopy(cfg))
            if render_path:
                plt.savefig(render_path, dpi=100, bbox_inches="tight")
            plt.close("all")
            r["renderable"] = True
        except Exception as e:
            r["error"] = f"render: {str(e)[:80]}"
            plt.close("all")

    checks = ["schema_complete","valid_charttype","has_data","has_title",
              "has_axes","renderable","has_annotations","data_consistency"]
    r["quality_score"] = sum(r[c] for c in checks)
    r["passed"] = r["quality_score"] >= 5
    return r

print("Validator + renderer ready")


Validator + renderer ready


In [4]:
import os, pandas as pd

ROOT_OUT = "master_run_results"
df = pd.read_csv(f"{ROOT_OUT}/master_pipeline_runs.csv")
print(f"Total CSV rows: {len(df)}")

configs_dir = f"{ROOT_OUT}/configs"
if not os.path.exists(configs_dir):
    print(f"⚠ configs/ folder DOES NOT EXIST — JSONs were never saved")
else:
    saved = sorted(os.listdir(configs_dir))
    print(f"JSON files on disk: {len(saved)}")
    print(f"First 15: {saved[:15]}")

renders_dir = f"{ROOT_OUT}/renders"
if os.path.exists(renders_dir):
    rendered = sorted(os.listdir(renders_dir))
    print(f"\nPNG files on disk: {len(rendered)}")

Total CSV rows: 54
JSON files on disk: 54
First 15: ['S0_run01.json', 'S0_run02.json', 'S0_run03.json', 'S0_run04.json', 'S0_run05.json', 'S0_run06.json', 'S0_run07.json', 'S0_run08.json', 'S0_run09.json', 'S0_run10.json', 'S1_run01.json', 'S1_run02.json', 'S1_run03.json', 'S1_run04.json', 'S1_run05.json']

PNG files on disk: 20


In [5]:
# Check what justify is actually returning
from generate_visualization import Agent
import inspect, json

# Look at justify_chart_type signature and source
print("=== justify_chart_type signature ===")
sig = inspect.signature(Agent.justify_chart_type)
print(f"Agent.justify_chart_type{sig}")

print("\n=== justify_chart_type source ===")
print(inspect.getsource(Agent.justify_chart_type))

print("\n=== What S4 saved (a justify-broken scenario) ===")
with open("master_run_results/configs/S4_run01.json") as f:
    saved = json.load(f)
print(json.dumps(saved, indent=2)[:800])

print("\n=== What S2 saved (a working scenario) ===")
with open("master_run_results/configs/S2_run01.json") as f:
    saved = json.load(f)
print(json.dumps(saved, indent=2)[:800])

# Check Agent's run method  
print("\n=== Does Agent have 'run' method? ===")
print(hasattr(Agent, "run"))
print("\n=== All Agent methods ===")
for name, method in inspect.getmembers(Agent, predicate=inspect.isfunction):
    if not name.startswith("_"):
        print(f"  Agent.{name}{inspect.signature(method)}")

=== justify_chart_type signature ===
Agent.justify_chart_type(self, analysis: dict, data: str) -> str

=== justify_chart_type source ===
    @retry(
        retry=retry_if_exception_type(ValidationError),
        stop=stop_after_attempt(4),
        wait=wait_exponential(multiplier=1, min=4, max=10),
    )

    def justify_chart_type(self,analysis: dict, data: str) -> str:
        """Create a Justification for Chart Types with a score"""
        charttypes = {chat_type.name for chat_type in ChartType}
        formatted_prompt = CREATE_CHART_TYPE_JUSTIFICATION_PROMPT.format(charttypes=charttypes,analysis=analysis, data = data)
        response = self.client.chat.completions.create(
            model=self.model,
            messages=[{"role": "user", "content": formatted_prompt}],
            response_model=ChartTypeJustification,
        )
        return response.chart_type.value


=== What S4 saved (a justify-broken scenario) ===
{
  "_error": "null"
}

=== What S2 saved (a working scen

## Cell 4 — Master loop (resume-safe)

In [22]:
# ═══════════════════════════════════════════════════════════════════
# CELL 4a — Batch 1: S0, S1, S2
# Wipes any prior runs of these 3 scenarios. Saves to master CSV.
# ═══════════════════════════════════════════════════════════════════
import os, time, json, csv

BATCH = ["S0", "S1", "S2"]
N_RUNS = 10
MASTER_CSV = f"{ROOT_OUT}/master_pipeline_runs.csv"

CSV_FIELDS = [
    "scenario","scenario_desc","run","timestamp","latency_s",
    "schema_complete","valid_charttype","has_data","has_title","has_axes",
    "renderable","has_annotations","data_consistency",
    "quality_score","passed","n_data_groups","n_annotations",
    "charttype","title","render_path","config_path","error"
]

# Ensure CSV exists with header
if not os.path.exists(MASTER_CSV):
    os.makedirs(ROOT_OUT, exist_ok=True)
    with open(MASTER_CSV, "w", newline="") as f:
        csv.DictWriter(f, fieldnames=CSV_FIELDS).writeheader()

# Wipe any prior rows for THIS BATCH only
import pandas as pd
if os.path.exists(MASTER_CSV):
    df = pd.read_csv(MASTER_CSV)
    df_keep = df[~df["scenario"].isin(BATCH)]
    df_keep.to_csv(MASTER_CSV, index=False)
    n_dropped = len(df) - len(df_keep)
    print(f"Wiped {n_dropped} prior rows for {BATCH} from CSV")

# Wipe prior config/render files for this batch
for sub in ["configs", "renders"]:
    p = f"{ROOT_OUT}/{sub}"
    os.makedirs(p, exist_ok=True)
    for f in os.listdir(p):
        if any(f.startswith(s + "_") for s in BATCH):
            os.remove(os.path.join(p, f))

print(f"Running batch: {BATCH}\n")

session_start = time.perf_counter()
attempted = 0
for sid in BATCH:
    desc, runner = SCENARIO_RUNNERS[sid]
    print(f"\n{'='*78}\n  {sid}: {desc}\n{'='*78}")
    for run_idx in range(1, N_RUNS+1):
        attempted += 1
        elapsed_min = (time.perf_counter() - session_start) / 60
        print(f"  Run {run_idx:02d}/{N_RUNS}  [{attempted}/{len(BATCH)*N_RUNS} | {elapsed_min:.1f}min]  ... ",
              end="", flush=True)

        ts = time.strftime("%Y-%m-%dT%H:%M:%S")
        t0 = time.perf_counter()
        cfg, err = None, None
        try:
            cfg = normalize_config(runner(QUESTION))
        except Exception as e:
            err = f"{type(e).__name__}: {str(e)[:120]}"
        lat = round(time.perf_counter()-t0, 3)

        config_path = f"{ROOT_OUT}/configs/{sid}_run{run_idx:02d}.json"
        if cfg is not None:
            try:
                with open(config_path,"w") as f:
                    json.dump(cfg, f, default=str, indent=2)
            except Exception as e:
                err = (err or "") + f" | save_cfg: {e}"
        else:
            with open(config_path,"w") as f:
                json.dump({"_error": err or "null"}, f)

        render_path = f"{ROOT_OUT}/renders/{sid}_run{run_idx:02d}.png"
        qr = validate(cfg, render=True, render_path=render_path) if cfg else {
            "schema_complete":False,"valid_charttype":False,"has_data":False,
            "has_title":False,"has_axes":False,"renderable":False,
            "has_annotations":False,"data_consistency":False,
            "quality_score":0,"passed":False,"error":err,
            "n_data_groups":0,"n_annotations":0,"title":"","charttype":"",
        }
        rendered_ok = qr.get("renderable") and os.path.exists(render_path)
        if not rendered_ok:
            try: os.remove(render_path)
            except: pass

        row = {
            "scenario": sid, "scenario_desc": desc, "run": run_idx, "timestamp": ts,
            "latency_s": lat,
            **{k: qr.get(k) for k in [
                "schema_complete","valid_charttype","has_data","has_title","has_axes",
                "renderable","has_annotations","data_consistency",
                "quality_score","passed","n_data_groups","n_annotations","charttype","title"
            ]},
            "render_path": render_path if rendered_ok else "",
            "config_path": config_path,
            "error": qr.get("error") or err or "",
        }
        with open(MASTER_CSV, "a", newline="") as f:
            csv.DictWriter(f, fieldnames=CSV_FIELDS).writerow(row)

        status = f"PASS {qr['quality_score']}/8" if qr.get("passed") else "FAIL"
        rsym = "OK" if rendered_ok else "--"
        title_short = (qr.get("title") or "")[:40]
        print(f"{lat:6.2f}s  {status} [{rsym}] chart={qr.get('charttype','?'):<11} "
              f"groups={qr.get('n_data_groups',0)} annos={qr.get('n_annotations',0)}  "
              f"title=\"{title_short}\"")

print(f"\n[Batch {BATCH} complete in {(time.perf_counter()-session_start)/60:.1f}min]")

Wiped 3 prior rows for ['S0', 'S1', 'S2'] from CSV
Running batch: ['S0', 'S1', 'S2']


  S0: 4-call: router+analyze+extract+justify [o4-mini]
  Run 01/10  [1/30 | 0.0min]  ... 

/Users/srujanakadambari/Desktop/ffresh thesis/data-to-visual/data-to-visual-nicos-branch/visualization_from_template.py:398: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


 38.11s  PASS 8/8 [OK] chart=line        groups=4 annos=6  title="Monatlicher Umsatz 2021–2024"
  Run 02/10  [2/30 | 0.6min]  ... 

/Users/srujanakadambari/Desktop/ffresh thesis/data-to-visual/data-to-visual-nicos-branch/visualization_from_template.py:398: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


 32.58s  PASS 8/8 [OK] chart=line        groups=4 annos=3  title="Umsatzentwicklung JVA 2021-2024"
  Run 03/10  [3/30 | 1.3min]  ... 

...

 53.98s  PASS 8/8 [OK] chart=line        groups=4 annos=5  title="Umsatz JVA von Teckentrup (Jan 2021 - De"
  Run 08/10  [28/30 | 19.0min]  ...  59.01s  PASS 8/8 [OK] chart=line        groups=1 annos=3  title="Monatliche Umsätze 2021–2024 (Segment JV"
  Run 09/10  [29/30 | 20.0min]  ... 

/Users/srujanakadambari/Desktop/ffresh thesis/data-to-visual/data-to-visual-nicos-branch/visualization_from_template.py:398: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/Users/srujanakadambari/Desktop/ffresh thesis/data-to-visual/data-to-visual-nicos-branch/visualization_from_template.py:398: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


 36.11s  PASS 8/8 [OK] chart=line        groups=4 annos=4  title="Teckentrup JVA Umsatz 2021–2024: Monatli"
  Run 10/10  [30/30 | 20.6min]  ... 

/Users/srujanakadambari/Desktop/ffresh thesis/data-to-visual/data-to-visual-nicos-branch/visualization_from_template.py:398: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


 43.33s  PASS 8/8 [OK] chart=column      groups=4 annos=2  title="Jahresvergleich Umsatz SD/CO JVA 2021-20"

[Batch ['S0', 'S1', 'S2'] complete in 21.4min]


In [12]:
# ═══════════════════════════════════════════════════════════════════
# CELL 4a — Batch 1: S0, S1, S2
# Wipes any prior runs of these 3 scenarios. Saves to master CSV.
# ═══════════════════════════════════════════════════════════════════
import os, time, json, csv

BATCH = ["S3", "S4", "S4b"]

N_RUNS = 10
MASTER_CSV = f"{ROOT_OUT}/master_pipeline_runs.csv"

CSV_FIELDS = [
    "scenario","scenario_desc","run","timestamp","latency_s",
    "schema_complete","valid_charttype","has_data","has_title","has_axes",
    "renderable","has_annotations","data_consistency",
    "quality_score","passed","n_data_groups","n_annotations",
    "charttype","title","render_path","config_path","error"
]

# Ensure CSV exists with header
if not os.path.exists(MASTER_CSV):
    os.makedirs(ROOT_OUT, exist_ok=True)
    with open(MASTER_CSV, "w", newline="") as f:
        csv.DictWriter(f, fieldnames=CSV_FIELDS).writeheader()

# Wipe any prior rows for THIS BATCH only
import pandas as pd
if os.path.exists(MASTER_CSV):
    df = pd.read_csv(MASTER_CSV)
    df_keep = df[~df["scenario"].isin(BATCH)]
    df_keep.to_csv(MASTER_CSV, index=False)
    n_dropped = len(df) - len(df_keep)
    print(f"Wiped {n_dropped} prior rows for {BATCH} from CSV")

# Wipe prior config/render files for this batch
for sub in ["configs", "renders"]:
    p = f"{ROOT_OUT}/{sub}"
    os.makedirs(p, exist_ok=True)
    for f in os.listdir(p):
        if any(f.startswith(s + "_") for s in BATCH):
            os.remove(os.path.join(p, f))

print(f"Running batch: {BATCH}\n")

session_start = time.perf_counter()
attempted = 0
for sid in BATCH:
    desc, runner = SCENARIO_RUNNERS[sid]
    print(f"\n{'='*78}\n  {sid}: {desc}\n{'='*78}")
    for run_idx in range(1, N_RUNS+1):
        attempted += 1
        elapsed_min = (time.perf_counter() - session_start) / 60
        print(f"  Run {run_idx:02d}/{N_RUNS}  [{attempted}/{len(BATCH)*N_RUNS} | {elapsed_min:.1f}min]  ... ",
              end="", flush=True)

        ts = time.strftime("%Y-%m-%dT%H:%M:%S")
        t0 = time.perf_counter()
        cfg, err = None, None
        try:
            cfg = normalize_config(runner(QUESTION))
        except Exception as e:
            err = f"{type(e).__name__}: {str(e)[:120]}"
        lat = round(time.perf_counter()-t0, 3)

        config_path = f"{ROOT_OUT}/configs/{sid}_run{run_idx:02d}.json"
        if cfg is not None:
            try:
                with open(config_path,"w") as f:
                    json.dump(cfg, f, default=str, indent=2)
            except Exception as e:
                err = (err or "") + f" | save_cfg: {e}"
        else:
            with open(config_path,"w") as f:
                json.dump({"_error": err or "null"}, f)

        render_path = f"{ROOT_OUT}/renders/{sid}_run{run_idx:02d}.png"
        qr = validate(cfg, render=True, render_path=render_path) if cfg else {
            "schema_complete":False,"valid_charttype":False,"has_data":False,
            "has_title":False,"has_axes":False,"renderable":False,
            "has_annotations":False,"data_consistency":False,
            "quality_score":0,"passed":False,"error":err,
            "n_data_groups":0,"n_annotations":0,"title":"","charttype":"",
        }
        rendered_ok = qr.get("renderable") and os.path.exists(render_path)
        if not rendered_ok:
            try: os.remove(render_path)
            except: pass

        row = {
            "scenario": sid, "scenario_desc": desc, "run": run_idx, "timestamp": ts,
            "latency_s": lat,
            **{k: qr.get(k) for k in [
                "schema_complete","valid_charttype","has_data","has_title","has_axes",
                "renderable","has_annotations","data_consistency",
                "quality_score","passed","n_data_groups","n_annotations","charttype","title"
            ]},
            "render_path": render_path if rendered_ok else "",
            "config_path": config_path,
            "error": qr.get("error") or err or "",
        }
        with open(MASTER_CSV, "a", newline="") as f:
            csv.DictWriter(f, fieldnames=CSV_FIELDS).writerow(row)

        status = f"PASS {qr['quality_score']}/8" if qr.get("passed") else "FAIL"
        rsym = "OK" if rendered_ok else "--"
        title_short = (qr.get("title") or "")[:40]
        print(f"{lat:6.2f}s  {status} [{rsym}] chart={qr.get('charttype','?'):<11} "
              f"groups={qr.get('n_data_groups',0)} annos={qr.get('n_annotations',0)}  "
              f"title=\"{title_short}\"")

elapsed_total_min = (time.perf_counter() - session_start) / 60
print(f"\n{'='*78}")
print(f"  ALL {attempted} RUNS COMPLETE in {elapsed_total_min:.1f} minutes")
print(f"{'='*78}")
print(f"  Master CSV: {MASTER_CSV}")
print(f"  Configs:    {ROOT_OUT}/configs/  ({attempted} JSONs)")
print(f"  Renders:    {ROOT_OUT}/renders/  (PNGs for successful renders)")

Wiped 0 prior rows for ['S3', 'S4', 'S4b'] from CSV
Running batch: ['S3', 'S4', 'S4b']


  S3: 1-call: single-shot extract [o4-mini]
  Run 01/10  [1/30 | 0.0min]  ...  18.72s  PASS 7/8 [--] chart=bar         groups=1 annos=4  title="Umsatz JVA Segment Teckentrup (2021-2024"
  Run 02/10  [2/30 | 0.3min]  ... 

/Users/srujanakadambari/Desktop/ffresh thesis/data-to-visual/data-to-visual-nicos-branch/visualization_from_template.py:398: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


 23.61s  PASS 8/8 [OK] chart=column      groups=1 annos=2  title="Jährlicher Umsatz Teckentrup im Segment "
  Run 03/10  [3/30 | 0.7min]  ...  16.33s  PASS 8/8 [OK] chart=column      groups=1 annos=4  title="Umsatz Teckentrup im Segment JVA (2021–2"
  Run 04/10  [4/30 | 1.0min]  ... 

/Users/srujanakadambari/Desktop/ffresh thesis/data-to-visual/data-to-visual-nicos-branch/visualization_from_template.py:398: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


 19.99s  PASS 7/8 [--] chart=line        groups=1 annos=4  title="Umsatz JVA 2021–2024"
  Run 05/10  [5/30 | 1.3min]  ...  17.90s  PASS 8/8 [OK] chart=column      groups=1 annos=4  title="Jährlicher Umsatz Teckentrup Segment JVA"
  Run 06/10  [6/30 | 1.6min]  ... 

...

 44.16s  PASS 8/8 [OK] chart=bar         groups=4 annos=8  title="Monthly Revenue from SD/CO Segment (2021"
  Run 09/10  [29/30 | 11.7min]  ... 

/Users/srujanakadambari/Desktop/ffresh thesis/data-to-visual/data-to-visual-nicos-branch/visualization_from_template.py:398: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


 46.77s  PASS 8/8 [OK] chart=line        groups=4 annos=4  title="Total Revenue Over the Years 2021-2024"
  Run 10/10  [30/30 | 12.4min]  ... 

/Users/srujanakadambari/Desktop/ffresh thesis/data-to-visual/data-to-visual-nicos-branch/visualization_from_template.py:398: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


 29.73s  PASS 8/8 [OK] chart=line        groups=4 annos=5  title="Monthly Sales Data from 2021 to 2024"

  ALL 30 RUNS COMPLETE in 12.9 minutes
  Master CSV: master_run_results/master_pipeline_runs.csv
  Configs:    master_run_results/configs/  (30 JSONs)
  Renders:    master_run_results/renders/  (PNGs for successful renders)


In [18]:
# ═══════════════════════════════════════════════════════════════════
# CELL 4a — Batch 1: S0, S1, S2
# Wipes any prior runs of these 3 scenarios. Saves to master CSV.
# ═══════════════════════════════════════════════════════════════════
import os, time, json, csv

BATCH = ["S5", "S5b", "S6"]

N_RUNS = 10
MASTER_CSV = f"{ROOT_OUT}/master_pipeline_runs.csv"

CSV_FIELDS = [
    "scenario","scenario_desc","run","timestamp","latency_s",
    "schema_complete","valid_charttype","has_data","has_title","has_axes",
    "renderable","has_annotations","data_consistency",
    "quality_score","passed","n_data_groups","n_annotations",
    "charttype","title","render_path","config_path","error"
]

# Ensure CSV exists with header
if not os.path.exists(MASTER_CSV):
    os.makedirs(ROOT_OUT, exist_ok=True)
    with open(MASTER_CSV, "w", newline="") as f:
        csv.DictWriter(f, fieldnames=CSV_FIELDS).writeheader()

# Wipe any prior rows for THIS BATCH only
import pandas as pd
if os.path.exists(MASTER_CSV):
    df = pd.read_csv(MASTER_CSV)
    df_keep = df[~df["scenario"].isin(BATCH)]
    df_keep.to_csv(MASTER_CSV, index=False)
    n_dropped = len(df) - len(df_keep)
    print(f"Wiped {n_dropped} prior rows for {BATCH} from CSV")

# Wipe prior config/render files for this batch
for sub in ["configs", "renders"]:
    p = f"{ROOT_OUT}/{sub}"
    os.makedirs(p, exist_ok=True)
    for f in os.listdir(p):
        if any(f.startswith(s + "_") for s in BATCH):
            os.remove(os.path.join(p, f))

print(f"Running batch: {BATCH}\n")

session_start = time.perf_counter()
attempted = 0
for sid in BATCH:
    desc, runner = SCENARIO_RUNNERS[sid]
    print(f"\n{'='*78}\n  {sid}: {desc}\n{'='*78}")
    for run_idx in range(1, N_RUNS+1):
        attempted += 1
        elapsed_min = (time.perf_counter() - session_start) / 60
        print(f"  Run {run_idx:02d}/{N_RUNS}  [{attempted}/{len(BATCH)*N_RUNS} | {elapsed_min:.1f}min]  ... ",
              end="", flush=True)

        ts = time.strftime("%Y-%m-%dT%H:%M:%S")
        t0 = time.perf_counter()
        cfg, err = None, None
        try:
            cfg = normalize_config(runner(QUESTION))
        except Exception as e:
            err = f"{type(e).__name__}: {str(e)[:120]}"
        lat = round(time.perf_counter()-t0, 3)

        config_path = f"{ROOT_OUT}/configs/{sid}_run{run_idx:02d}.json"
        if cfg is not None:
            try:
                with open(config_path,"w") as f:
                    json.dump(cfg, f, default=str, indent=2)
            except Exception as e:
                err = (err or "") + f" | save_cfg: {e}"
        else:
            with open(config_path,"w") as f:
                json.dump({"_error": err or "null"}, f)

        render_path = f"{ROOT_OUT}/renders/{sid}_run{run_idx:02d}.png"
        qr = validate(cfg, render=True, render_path=render_path) if cfg else {
            "schema_complete":False,"valid_charttype":False,"has_data":False,
            "has_title":False,"has_axes":False,"renderable":False,
            "has_annotations":False,"data_consistency":False,
            "quality_score":0,"passed":False,"error":err,
            "n_data_groups":0,"n_annotations":0,"title":"","charttype":"",
        }
        rendered_ok = qr.get("renderable") and os.path.exists(render_path)
        if not rendered_ok:
            try: os.remove(render_path)
            except: pass

        row = {
            "scenario": sid, "scenario_desc": desc, "run": run_idx, "timestamp": ts,
            "latency_s": lat,
            **{k: qr.get(k) for k in [
                "schema_complete","valid_charttype","has_data","has_title","has_axes",
                "renderable","has_annotations","data_consistency",
                "quality_score","passed","n_data_groups","n_annotations","charttype","title"
            ]},
            "render_path": render_path if rendered_ok else "",
            "config_path": config_path,
            "error": qr.get("error") or err or "",
        }
        with open(MASTER_CSV, "a", newline="") as f:
            csv.DictWriter(f, fieldnames=CSV_FIELDS).writerow(row)

        status = f"PASS {qr['quality_score']}/8" if qr.get("passed") else "FAIL"
        rsym = "OK" if rendered_ok else "--"
        title_short = (qr.get("title") or "")[:40]
        print(f"{lat:6.2f}s  {status} [{rsym}] chart={qr.get('charttype','?'):<11} "
              f"groups={qr.get('n_data_groups',0)} annos={qr.get('n_annotations',0)}  "
              f"title=\"{title_short}\"")

elapsed_total_min = (time.perf_counter() - session_start) / 60
print(f"\n{'='*78}")
print(f"  ALL {attempted} RUNS COMPLETE in {elapsed_total_min:.1f} minutes")
print(f"{'='*78}")
print(f"  Master CSV: {MASTER_CSV}")
print(f"  Configs:    {ROOT_OUT}/configs/  ({attempted} JSONs)")
print(f"  Renders:    {ROOT_OUT}/renders/  (PNGs for successful renders)")

Wiped 30 prior rows for ['S5', 'S5b', 'S6'] from CSV
Running batch: ['S5', 'S5b', 'S6']


  S5: 2-call: analyze[o4-mini]+extract[4o-mini]
  Run 01/10  [1/30 | 0.0min]  ... 

/Users/srujanakadambari/Desktop/ffresh thesis/data-to-visual/data-to-visual-nicos-branch/visualization_from_template.py:398: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


 23.88s  PASS 8/8 [OK] chart=bar         groups=1 annos=4  title="Gesamt-Umsatz pro Jahr (Segment JVA)"
  Run 02/10  [2/30 | 0.4min]  ... 

/Users/srujanakadambari/Desktop/ffresh thesis/data-to-visual/data-to-visual-nicos-branch/visualization_from_template.py:398: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


 50.39s  PASS 8/8 [OK] chart=line        groups=4 annos=4  title="Monatsumsatz 2021–2024"
  Run 03/10  [3/30 | 1.3min]  ... 

...

 18.28s  PASS 8/8 [OK] chart=line        groups=4 annos=4  title="Umsatzanalyse Teckentrup 2021-2024"
  Run 07/10  [27/30 | 12.3min]  ...  24.76s  PASS 8/8 [OK] chart=line        groups=4 annos=2  title="Umsatz Teckentrup JVA 2021-2024"
  Run 08/10  [28/30 | 12.8min]  ... 

/Users/srujanakadambari/Desktop/ffresh thesis/data-to-visual/data-to-visual-nicos-branch/visualization_from_template.py:398: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/Users/srujanakadambari/Desktop/ffresh thesis/data-to-visual/data-to-visual-nicos-branch/visualization_from_template.py:398: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


 28.69s  PASS 8/8 [OK] chart=line        groups=4 annos=3  title="Umsatz von Teckentrup im Segment JVA (20"
  Run 09/10  [29/30 | 13.2min]  ...  14.58s  PASS 7/8 [--] chart=bar         groups=4 annos=3  title="Umsatz Teckentrup im Segment JVA (2021-2"
  Run 10/10  [30/30 | 13.5min]  ... 

/Users/srujanakadambari/Desktop/ffresh thesis/data-to-visual/data-to-visual-nicos-branch/visualization_from_template.py:398: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


 14.55s  PASS 8/8 [OK] chart=bar         groups=4 annos=3  title="Umsatz von Teckentrup im Segment JVA (20"

  ALL 30 RUNS COMPLETE in 13.7 minutes
  Master CSV: master_run_results/master_pipeline_runs.csv
  Configs:    master_run_results/configs/  (30 JSONs)
  Renders:    master_run_results/renders/  (PNGs for successful renders)


In [14]:
# ═══════════════════════════════════════════════════════════════════
# CELL 4a — Batch 1: S0, S1, S2
# Wipes any prior runs of these 3 scenarios. Saves to master CSV.
# ═══════════════════════════════════════════════════════════════════
import os, time, json, csv

BATCH = ["S7", "S8", "S9"]
N_RUNS = 10
MASTER_CSV = f"{ROOT_OUT}/master_pipeline_runs.csv"

CSV_FIELDS = [
    "scenario","scenario_desc","run","timestamp","latency_s",
    "schema_complete","valid_charttype","has_data","has_title","has_axes",
    "renderable","has_annotations","data_consistency",
    "quality_score","passed","n_data_groups","n_annotations",
    "charttype","title","render_path","config_path","error"
]

# Ensure CSV exists with header
if not os.path.exists(MASTER_CSV):
    os.makedirs(ROOT_OUT, exist_ok=True)
    with open(MASTER_CSV, "w", newline="") as f:
        csv.DictWriter(f, fieldnames=CSV_FIELDS).writeheader()

# Wipe any prior rows for THIS BATCH only
import pandas as pd
if os.path.exists(MASTER_CSV):
    df = pd.read_csv(MASTER_CSV)
    df_keep = df[~df["scenario"].isin(BATCH)]
    df_keep.to_csv(MASTER_CSV, index=False)
    n_dropped = len(df) - len(df_keep)
    print(f"Wiped {n_dropped} prior rows for {BATCH} from CSV")

# Wipe prior config/render files for this batch
for sub in ["configs", "renders"]:
    p = f"{ROOT_OUT}/{sub}"
    os.makedirs(p, exist_ok=True)
    for f in os.listdir(p):
        if any(f.startswith(s + "_") for s in BATCH):
            os.remove(os.path.join(p, f))

print(f"Running batch: {BATCH}\n")

session_start = time.perf_counter()
attempted = 0
for sid in BATCH:
    desc, runner = SCENARIO_RUNNERS[sid]
    print(f"\n{'='*78}\n  {sid}: {desc}\n{'='*78}")
    for run_idx in range(1, N_RUNS+1):
        attempted += 1
        elapsed_min = (time.perf_counter() - session_start) / 60
        print(f"  Run {run_idx:02d}/{N_RUNS}  [{attempted}/{len(BATCH)*N_RUNS} | {elapsed_min:.1f}min]  ... ",
              end="", flush=True)

        ts = time.strftime("%Y-%m-%dT%H:%M:%S")
        t0 = time.perf_counter()
        cfg, err = None, None
        try:
            cfg = normalize_config(runner(QUESTION))
        except Exception as e:
            err = f"{type(e).__name__}: {str(e)[:120]}"
        lat = round(time.perf_counter()-t0, 3)

        config_path = f"{ROOT_OUT}/configs/{sid}_run{run_idx:02d}.json"
        if cfg is not None:
            try:
                with open(config_path,"w") as f:
                    json.dump(cfg, f, default=str, indent=2)
            except Exception as e:
                err = (err or "") + f" | save_cfg: {e}"
        else:
            with open(config_path,"w") as f:
                json.dump({"_error": err or "null"}, f)

        render_path = f"{ROOT_OUT}/renders/{sid}_run{run_idx:02d}.png"
        qr = validate(cfg, render=True, render_path=render_path) if cfg else {
            "schema_complete":False,"valid_charttype":False,"has_data":False,
            "has_title":False,"has_axes":False,"renderable":False,
            "has_annotations":False,"data_consistency":False,
            "quality_score":0,"passed":False,"error":err,
            "n_data_groups":0,"n_annotations":0,"title":"","charttype":"",
        }
        rendered_ok = qr.get("renderable") and os.path.exists(render_path)
        if not rendered_ok:
            try: os.remove(render_path)
            except: pass

        row = {
            "scenario": sid, "scenario_desc": desc, "run": run_idx, "timestamp": ts,
            "latency_s": lat,
            **{k: qr.get(k) for k in [
                "schema_complete","valid_charttype","has_data","has_title","has_axes",
                "renderable","has_annotations","data_consistency",
                "quality_score","passed","n_data_groups","n_annotations","charttype","title"
            ]},
            "render_path": render_path if rendered_ok else "",
            "config_path": config_path,
            "error": qr.get("error") or err or "",
        }
        with open(MASTER_CSV, "a", newline="") as f:
            csv.DictWriter(f, fieldnames=CSV_FIELDS).writerow(row)

        status = f"PASS {qr['quality_score']}/8" if qr.get("passed") else "FAIL"
        rsym = "OK" if rendered_ok else "--"
        title_short = (qr.get("title") or "")[:40]
        print(f"{lat:6.2f}s  {status} [{rsym}] chart={qr.get('charttype','?'):<11} "
              f"groups={qr.get('n_data_groups',0)} annos={qr.get('n_annotations',0)}  "
              f"title=\"{title_short}\"")

elapsed_total_min = (time.perf_counter() - session_start) / 60
print(f"\n{'='*78}")
print(f"  ALL {attempted} RUNS COMPLETE in {elapsed_total_min:.1f} minutes")
print(f"{'='*78}")
print(f"  Master CSV: {MASTER_CSV}")
print(f"  Configs:    {ROOT_OUT}/configs/  ({attempted} JSONs)")
print(f"  Renders:    {ROOT_OUT}/renders/  (PNGs for successful renders)")

Wiped 0 prior rows for ['S7', 'S8', 'S9'] from CSV
Running batch: ['S7', 'S8', 'S9']


  S7: 3-call: analyze[o3]+extract[4o-mini]+justify
  Run 01/10  [1/30 | 0.0min]  ... 

/Users/srujanakadambari/Desktop/ffresh thesis/data-to-visual/data-to-visual-nicos-branch/visualization_from_template.py:398: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


675.18s  PASS 8/8 [OK] chart=column      groups=4 annos=4  title="Jahresumsätze 2021–2024"
  Run 02/10  [2/30 | 11.3min]  ...  49.40s  PASS 8/8 [OK] chart=column      groups=1 annos=4  title="Jahresumsätze Segment JVA (Teckentrup) 2"
  Run 03/10  [3/30 | 12.1min]  ... 

/Users/srujanakadambari/Desktop/ffresh thesis/data-to-visual/data-to-visual-nicos-branch/visualization_from_template.py:398: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


 32.77s  PASS 7/8 [--] chart=stackedbar  groups=4 annos=4  title="Total Umsatz SD/CO (Segment JVA) 2021-20"
  Run 04/10  [4/30 | 12.6min]  ... 

...

 13.58s  PASS 8/8 [OK] chart=bar         groups=1 annos=2  title="Umsatz von Teckentrup im Segment JVA (20"
  Run 08/10  [28/30 | 48.1min]  ... 

/Users/srujanakadambari/Desktop/ffresh thesis/data-to-visual/data-to-visual-nicos-branch/visualization_from_template.py:398: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


  6.11s  PASS 8/8 [OK] chart=line        groups=12 annos=2  title="Umsatz von Teckentrup im Segment JVA (20"
  Run 09/10  [29/30 | 48.2min]  ... 

/Users/srujanakadambari/Desktop/ffresh thesis/data-to-visual/data-to-visual-nicos-branch/visualization_from_template.py:398: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


  5.54s  PASS 8/8 [OK] chart=line        groups=4 annos=3  title="Umsatz von Teckentrup im Segment JVA (20"
  Run 10/10  [30/30 | 48.3min]  ...   6.22s  PASS 6/8 [--] chart=line        groups=1 annos=4  title="Umsatz von Teckentrup im Segment JVA"

  ALL 30 RUNS COMPLETE in 48.4 minutes
  Master CSV: master_run_results/master_pipeline_runs.csv
  Configs:    master_run_results/configs/  (30 JSONs)
  Renders:    master_run_results/renders/  (PNGs for successful renders)


In [15]:
# ═══════════════════════════════════════════════════════════════════
# CELL 4a — Batch 1: S0, S1, S2
# Wipes any prior runs of these 3 scenarios. Saves to master CSV.
# ═══════════════════════════════════════════════════════════════════
import os, time, json, csv

BATCH = ["SA1", "SA2", "SA3"]

N_RUNS = 10
MASTER_CSV = f"{ROOT_OUT}/master_pipeline_runs.csv"

CSV_FIELDS = [
    "scenario","scenario_desc","run","timestamp","latency_s",
    "schema_complete","valid_charttype","has_data","has_title","has_axes",
    "renderable","has_annotations","data_consistency",
    "quality_score","passed","n_data_groups","n_annotations",
    "charttype","title","render_path","config_path","error"
]

# Ensure CSV exists with header
if not os.path.exists(MASTER_CSV):
    os.makedirs(ROOT_OUT, exist_ok=True)
    with open(MASTER_CSV, "w", newline="") as f:
        csv.DictWriter(f, fieldnames=CSV_FIELDS).writeheader()

# Wipe any prior rows for THIS BATCH only
import pandas as pd
if os.path.exists(MASTER_CSV):
    df = pd.read_csv(MASTER_CSV)
    df_keep = df[~df["scenario"].isin(BATCH)]
    df_keep.to_csv(MASTER_CSV, index=False)
    n_dropped = len(df) - len(df_keep)
    print(f"Wiped {n_dropped} prior rows for {BATCH} from CSV")

# Wipe prior config/render files for this batch
for sub in ["configs", "renders"]:
    p = f"{ROOT_OUT}/{sub}"
    os.makedirs(p, exist_ok=True)
    for f in os.listdir(p):
        if any(f.startswith(s + "_") for s in BATCH):
            os.remove(os.path.join(p, f))

print(f"Running batch: {BATCH}\n")

session_start = time.perf_counter()
attempted = 0
for sid in BATCH:
    desc, runner = SCENARIO_RUNNERS[sid]
    print(f"\n{'='*78}\n  {sid}: {desc}\n{'='*78}")
    for run_idx in range(1, N_RUNS+1):
        attempted += 1
        elapsed_min = (time.perf_counter() - session_start) / 60
        print(f"  Run {run_idx:02d}/{N_RUNS}  [{attempted}/{len(BATCH)*N_RUNS} | {elapsed_min:.1f}min]  ... ",
              end="", flush=True)

        ts = time.strftime("%Y-%m-%dT%H:%M:%S")
        t0 = time.perf_counter()
        cfg, err = None, None
        try:
            cfg = normalize_config(runner(QUESTION))
        except Exception as e:
            err = f"{type(e).__name__}: {str(e)[:120]}"
        lat = round(time.perf_counter()-t0, 3)

        config_path = f"{ROOT_OUT}/configs/{sid}_run{run_idx:02d}.json"
        if cfg is not None:
            try:
                with open(config_path,"w") as f:
                    json.dump(cfg, f, default=str, indent=2)
            except Exception as e:
                err = (err or "") + f" | save_cfg: {e}"
        else:
            with open(config_path,"w") as f:
                json.dump({"_error": err or "null"}, f)

        render_path = f"{ROOT_OUT}/renders/{sid}_run{run_idx:02d}.png"
        qr = validate(cfg, render=True, render_path=render_path) if cfg else {
            "schema_complete":False,"valid_charttype":False,"has_data":False,
            "has_title":False,"has_axes":False,"renderable":False,
            "has_annotations":False,"data_consistency":False,
            "quality_score":0,"passed":False,"error":err,
            "n_data_groups":0,"n_annotations":0,"title":"","charttype":"",
        }
        rendered_ok = qr.get("renderable") and os.path.exists(render_path)
        if not rendered_ok:
            try: os.remove(render_path)
            except: pass

        row = {
            "scenario": sid, "scenario_desc": desc, "run": run_idx, "timestamp": ts,
            "latency_s": lat,
            **{k: qr.get(k) for k in [
                "schema_complete","valid_charttype","has_data","has_title","has_axes",
                "renderable","has_annotations","data_consistency",
                "quality_score","passed","n_data_groups","n_annotations","charttype","title"
            ]},
            "render_path": render_path if rendered_ok else "",
            "config_path": config_path,
            "error": qr.get("error") or err or "",
        }
        with open(MASTER_CSV, "a", newline="") as f:
            csv.DictWriter(f, fieldnames=CSV_FIELDS).writerow(row)

        status = f"PASS {qr['quality_score']}/8" if qr.get("passed") else "FAIL"
        rsym = "OK" if rendered_ok else "--"
        title_short = (qr.get("title") or "")[:40]
        print(f"{lat:6.2f}s  {status} [{rsym}] chart={qr.get('charttype','?'):<11} "
              f"groups={qr.get('n_data_groups',0)} annos={qr.get('n_annotations',0)}  "
              f"title=\"{title_short}\"")

elapsed_total_min = (time.perf_counter() - session_start) / 60
print(f"\n{'='*78}")
print(f"  ALL {attempted} RUNS COMPLETE in {elapsed_total_min:.1f} minutes")
print(f"{'='*78}")
print(f"  Master CSV: {MASTER_CSV}")
print(f"  Configs:    {ROOT_OUT}/configs/  ({attempted} JSONs)")
print(f"  Renders:    {ROOT_OUT}/renders/  (PNGs for successful renders)")

Wiped 0 prior rows for ['SA1', 'SA2', 'SA3'] from CSV
Running batch: ['SA1', 'SA2', 'SA3']


  SA1: 4-call: router+analyze+extract+justify [o4-mini]
  Run 01/10  [1/30 | 0.0min]  ... 

/Users/srujanakadambari/Desktop/ffresh thesis/data-to-visual/data-to-visual-nicos-branch/visualization_from_template.py:398: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


 55.15s  PASS 8/8 [OK] chart=line        groups=4 annos=3  title="Monatlicher Umsatztrend JVA (2021–2024)"
  Run 02/10  [2/30 | 0.9min]  ... 

/Users/srujanakadambari/Desktop/ffresh thesis/data-to-visual/data-to-visual-nicos-branch/visualization_from_template.py:398: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


 56.83s  PASS 8/8 [OK] chart=line        groups=1 annos=1  title="Annual Sales (2021–2024)"
  Run 03/10  [3/30 | 1.9min]  ... 

...

 23.71s  PASS 8/8 [OK] chart=line        groups=4 annos=4  title="Monthly Sales of Teckentrup in JVA Segme"
  Run 08/10  [28/30 | 18.1min]  ... 

/Users/srujanakadambari/Desktop/ffresh thesis/data-to-visual/data-to-visual-nicos-branch/visualization_from_template.py:398: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


 16.89s  PASS 8/8 [OK] chart=line        groups=4 annos=4  title="Monthly Revenue Trends from 2021 to 2024"
  Run 09/10  [29/30 | 18.3min]  ... 

/Users/srujanakadambari/Desktop/ffresh thesis/data-to-visual/data-to-visual-nicos-branch/visualization_from_template.py:398: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


 18.68s  PASS 8/8 [OK] chart=bar         groups=1 annos=6  title="Monthly Sales Revenue for Teckentrup in "
  Run 10/10  [30/30 | 18.7min]  ...  27.58s  PASS 7/8 [--] chart=bar         groups=4 annos=8  title="Monthly Sales Data from 2021 to 2024"

  ALL 30 RUNS COMPLETE in 19.2 minutes
  Master CSV: master_run_results/master_pipeline_runs.csv
  Configs:    master_run_results/configs/  (30 JSONs)
  Renders:    master_run_results/renders/  (PNGs for successful renders)


In [20]:
# ═══════════════════════════════════════════════════════════════════
# CELL 4a — Batch 1: S0, S1, S2
# Wipes any prior runs of these 3 scenarios. Saves to master CSV.
# ═══════════════════════════════════════════════════════════════════
import os, time, json, csv

BATCH = ["S7"]
N_RUNS = 10
MASTER_CSV = f"{ROOT_OUT}/master_pipeline_runs.csv"

CSV_FIELDS = [
    "scenario","scenario_desc","run","timestamp","latency_s",
    "schema_complete","valid_charttype","has_data","has_title","has_axes",
    "renderable","has_annotations","data_consistency",
    "quality_score","passed","n_data_groups","n_annotations",
    "charttype","title","render_path","config_path","error"
]

# Ensure CSV exists with header
if not os.path.exists(MASTER_CSV):
    os.makedirs(ROOT_OUT, exist_ok=True)
    with open(MASTER_CSV, "w", newline="") as f:
        csv.DictWriter(f, fieldnames=CSV_FIELDS).writeheader()

# Wipe any prior rows for THIS BATCH only
import pandas as pd
if os.path.exists(MASTER_CSV):
    df = pd.read_csv(MASTER_CSV)
    df_keep = df[~df["scenario"].isin(BATCH)]
    df_keep.to_csv(MASTER_CSV, index=False)
    n_dropped = len(df) - len(df_keep)
    print(f"Wiped {n_dropped} prior rows for {BATCH} from CSV")

# Wipe prior config/render files for this batch
for sub in ["configs", "renders"]:
    p = f"{ROOT_OUT}/{sub}"
    os.makedirs(p, exist_ok=True)
    for f in os.listdir(p):
        if any(f.startswith(s + "_") for s in BATCH):
            os.remove(os.path.join(p, f))

print(f"Running batch: {BATCH}\n")

session_start = time.perf_counter()
attempted = 0
for sid in BATCH:
    desc, runner = SCENARIO_RUNNERS[sid]
    print(f"\n{'='*78}\n  {sid}: {desc}\n{'='*78}")
    for run_idx in range(1, N_RUNS+1):
        attempted += 1
        elapsed_min = (time.perf_counter() - session_start) / 60
        print(f"  Run {run_idx:02d}/{N_RUNS}  [{attempted}/{len(BATCH)*N_RUNS} | {elapsed_min:.1f}min]  ... ",
              end="", flush=True)

        ts = time.strftime("%Y-%m-%dT%H:%M:%S")
        t0 = time.perf_counter()
        cfg, err = None, None
        try:
            cfg = normalize_config(runner(QUESTION))
        except Exception as e:
            err = f"{type(e).__name__}: {str(e)[:120]}"
        lat = round(time.perf_counter()-t0, 3)

        config_path = f"{ROOT_OUT}/configs/{sid}_run{run_idx:02d}.json"
        if cfg is not None:
            try:
                with open(config_path,"w") as f:
                    json.dump(cfg, f, default=str, indent=2)
            except Exception as e:
                err = (err or "") + f" | save_cfg: {e}"
        else:
            with open(config_path,"w") as f:
                json.dump({"_error": err or "null"}, f)

        render_path = f"{ROOT_OUT}/renders/{sid}_run{run_idx:02d}.png"
        qr = validate(cfg, render=True, render_path=render_path) if cfg else {
            "schema_complete":False,"valid_charttype":False,"has_data":False,
            "has_title":False,"has_axes":False,"renderable":False,
            "has_annotations":False,"data_consistency":False,
            "quality_score":0,"passed":False,"error":err,
            "n_data_groups":0,"n_annotations":0,"title":"","charttype":"",
        }
        rendered_ok = qr.get("renderable") and os.path.exists(render_path)
        if not rendered_ok:
            try: os.remove(render_path)
            except: pass

        row = {
            "scenario": sid, "scenario_desc": desc, "run": run_idx, "timestamp": ts,
            "latency_s": lat,
            **{k: qr.get(k) for k in [
                "schema_complete","valid_charttype","has_data","has_title","has_axes",
                "renderable","has_annotations","data_consistency",
                "quality_score","passed","n_data_groups","n_annotations","charttype","title"
            ]},
            "render_path": render_path if rendered_ok else "",
            "config_path": config_path,
            "error": qr.get("error") or err or "",
        }
        with open(MASTER_CSV, "a", newline="") as f:
            csv.DictWriter(f, fieldnames=CSV_FIELDS).writerow(row)

        status = f"PASS {qr['quality_score']}/8" if qr.get("passed") else "FAIL"
        rsym = "OK" if rendered_ok else "--"
        title_short = (qr.get("title") or "")[:40]
        print(f"{lat:6.2f}s  {status} [{rsym}] chart={qr.get('charttype','?'):<11} "
              f"groups={qr.get('n_data_groups',0)} annos={qr.get('n_annotations',0)}  "
              f"title=\"{title_short}\"")

elapsed_total_min = (time.perf_counter() - session_start) / 60
print(f"\n{'='*78}")
print(f"  ALL {attempted} RUNS COMPLETE in {elapsed_total_min:.1f} minutes")
print(f"{'='*78}")
print(f"  Master CSV: {MASTER_CSV}")
print(f"  Configs:    {ROOT_OUT}/configs/  ({attempted} JSONs)")
print(f"  Renders:    {ROOT_OUT}/renders/  (PNGs for successful renders)")

Wiped 10 prior rows for ['S7'] from CSV
Running batch: ['S7']


  S7: 3-call: analyze[o3]+extract[4o-mini]+justify
  Run 01/10  [1/10 | 0.0min]  ... 

/Users/srujanakadambari/Desktop/ffresh thesis/data-to-visual/data-to-visual-nicos-branch/visualization_from_template.py:398: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


 36.38s  PASS 8/8 [OK] chart=stackedbar  groups=4 annos=4  title="Monatliche Umsatzentwicklung (2021-2024)"
  Run 02/10  [2/10 | 0.6min]  ...  33.31s  PASS 7/8 [--] chart=line        groups=1 annos=5  title="Jahressummen (Umsatz SD/CO) 2021-2024 – "
  Run 03/10  [3/10 | 1.2min]  ...  36.86s  PASS 6/8 [--] chart=column      groups=1 annos=3  title="Jahresumsätze 2021–2024"
  Run 04/10  [4/10 | 1.8min]  ...  34.35s  PASS 7/8 [--] chart=stackedbar  groups=4 annos=7  title="Monatliche JVA-Umsätze der Firma Teckent"
  Run 05/10  [5/10 | 2.4min]  ... 

/Users/srujanakadambari/Desktop/ffresh thesis/data-to-visual/data-to-visual-nicos-branch/visualization_from_template.py:398: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


 31.91s  PASS 8/8 [OK] chart=line        groups=4 annos=5  title="Monthly Revenue Analysis from 2021 to 20"
  Run 06/10  [6/10 | 2.9min]  ...  37.41s  PASS 6/8 [--] chart=column      groups=1 annos=6  title="Gesamtumsatz 2021-2024 (Segment JVA)"
  Run 07/10  [7/10 | 3.5min]  ...  45.95s  PASS 6/8 [--] chart=bar         groups=1 annos=8  title="Umsatzentwicklung 2021-2024"
  Run 08/10  [8/10 | 4.3min]  ...  33.88s  PASS 6/8 [--] chart=line        groups=1 annos=3  title="Umsatzentwicklung Teckentrup (2021-2024)"
  Run 09/10  [9/10 | 4.9min]  ... 

/Users/srujanakadambari/Desktop/ffresh thesis/data-to-visual/data-to-visual-nicos-branch/visualization_from_template.py:398: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


 43.40s  PASS 8/8 [OK] chart=column      groups=4 annos=13  title="Jahresumsätze Segment JVA (Teckentrup) 2"
  Run 10/10  [10/10 | 5.6min]  ... 

/Users/srujanakadambari/Desktop/ffresh thesis/data-to-visual/data-to-visual-nicos-branch/visualization_from_template.py:398: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


 42.84s  PASS 8/8 [OK] chart=bar         groups=4 annos=6  title="Teckentrup – Umsatz Segment JVA 2021-202"

  ALL 10 RUNS COMPLETE in 6.3 minutes
  Master CSV: master_run_results/master_pipeline_runs.csv
  Configs:    master_run_results/configs/  (10 JSONs)
  Renders:    master_run_results/renders/  (PNGs for successful renders)


In [16]:
# ═══════════════════════════════════════════════════════════════════
# CELL 4a — Batch 1: S0, S1, S2
# Wipes any prior runs of these 3 scenarios. Saves to master CSV.
# ═══════════════════════════════════════════════════════════════════
import os, time, json, csv

BATCH = ["SA4", "SA5", "SA6"]
N_RUNS = 10
MASTER_CSV = f"{ROOT_OUT}/master_pipeline_runs.csv"

CSV_FIELDS = [
    "scenario","scenario_desc","run","timestamp","latency_s",
    "schema_complete","valid_charttype","has_data","has_title","has_axes",
    "renderable","has_annotations","data_consistency",
    "quality_score","passed","n_data_groups","n_annotations",
    "charttype","title","render_path","config_path","error"
]

# Ensure CSV exists with header
if not os.path.exists(MASTER_CSV):
    os.makedirs(ROOT_OUT, exist_ok=True)
    with open(MASTER_CSV, "w", newline="") as f:
        csv.DictWriter(f, fieldnames=CSV_FIELDS).writeheader()

# Wipe any prior rows for THIS BATCH only
import pandas as pd
if os.path.exists(MASTER_CSV):
    df = pd.read_csv(MASTER_CSV)
    df_keep = df[~df["scenario"].isin(BATCH)]
    df_keep.to_csv(MASTER_CSV, index=False)
    n_dropped = len(df) - len(df_keep)
    print(f"Wiped {n_dropped} prior rows for {BATCH} from CSV")

# Wipe prior config/render files for this batch
for sub in ["configs", "renders"]:
    p = f"{ROOT_OUT}/{sub}"
    os.makedirs(p, exist_ok=True)
    for f in os.listdir(p):
        if any(f.startswith(s + "_") for s in BATCH):
            os.remove(os.path.join(p, f))

print(f"Running batch: {BATCH}\n")

session_start = time.perf_counter()
attempted = 0
for sid in BATCH:
    desc, runner = SCENARIO_RUNNERS[sid]
    print(f"\n{'='*78}\n  {sid}: {desc}\n{'='*78}")
    for run_idx in range(1, N_RUNS+1):
        attempted += 1
        elapsed_min = (time.perf_counter() - session_start) / 60
        print(f"  Run {run_idx:02d}/{N_RUNS}  [{attempted}/{len(BATCH)*N_RUNS} | {elapsed_min:.1f}min]  ... ",
              end="", flush=True)

        ts = time.strftime("%Y-%m-%dT%H:%M:%S")
        t0 = time.perf_counter()
        cfg, err = None, None
        try:
            cfg = normalize_config(runner(QUESTION))
        except Exception as e:
            err = f"{type(e).__name__}: {str(e)[:120]}"
        lat = round(time.perf_counter()-t0, 3)

        config_path = f"{ROOT_OUT}/configs/{sid}_run{run_idx:02d}.json"
        if cfg is not None:
            try:
                with open(config_path,"w") as f:
                    json.dump(cfg, f, default=str, indent=2)
            except Exception as e:
                err = (err or "") + f" | save_cfg: {e}"
        else:
            with open(config_path,"w") as f:
                json.dump({"_error": err or "null"}, f)

        render_path = f"{ROOT_OUT}/renders/{sid}_run{run_idx:02d}.png"
        qr = validate(cfg, render=True, render_path=render_path) if cfg else {
            "schema_complete":False,"valid_charttype":False,"has_data":False,
            "has_title":False,"has_axes":False,"renderable":False,
            "has_annotations":False,"data_consistency":False,
            "quality_score":0,"passed":False,"error":err,
            "n_data_groups":0,"n_annotations":0,"title":"","charttype":"",
        }
        rendered_ok = qr.get("renderable") and os.path.exists(render_path)
        if not rendered_ok:
            try: os.remove(render_path)
            except: pass

        row = {
            "scenario": sid, "scenario_desc": desc, "run": run_idx, "timestamp": ts,
            "latency_s": lat,
            **{k: qr.get(k) for k in [
                "schema_complete","valid_charttype","has_data","has_title","has_axes",
                "renderable","has_annotations","data_consistency",
                "quality_score","passed","n_data_groups","n_annotations","charttype","title"
            ]},
            "render_path": render_path if rendered_ok else "",
            "config_path": config_path,
            "error": qr.get("error") or err or "",
        }
        with open(MASTER_CSV, "a", newline="") as f:
            csv.DictWriter(f, fieldnames=CSV_FIELDS).writerow(row)

        status = f"PASS {qr['quality_score']}/8" if qr.get("passed") else "FAIL"
        rsym = "OK" if rendered_ok else "--"
        title_short = (qr.get("title") or "")[:40]
        print(f"{lat:6.2f}s  {status} [{rsym}] chart={qr.get('charttype','?'):<11} "
              f"groups={qr.get('n_data_groups',0)} annos={qr.get('n_annotations',0)}  "
              f"title=\"{title_short}\"")

elapsed_total_min = (time.perf_counter() - session_start) / 60
print(f"\n{'='*78}")
print(f"  ALL {attempted} RUNS COMPLETE in {elapsed_total_min:.1f} minutes")
print(f"{'='*78}")
print(f"  Master CSV: {MASTER_CSV}")
print(f"  Configs:    {ROOT_OUT}/configs/  ({attempted} JSONs)")
print(f"  Renders:    {ROOT_OUT}/renders/  (PNGs for successful renders)")

Wiped 0 prior rows for ['SA4', 'SA5', 'SA6'] from CSV
Running batch: ['SA4', 'SA5', 'SA6']


  SA4: 3-call: router+analyze[gpt-4o]+extract[4o-mini]
  Run 01/10  [1/30 | 0.0min]  ...  30.96s  PASS 8/8 [OK] chart=line        groups=4 annos=6  title="Monthly Revenue Trend for Teckentrup (20"
  Run 02/10  [2/30 | 0.5min]  ... 

/Users/srujanakadambari/Desktop/ffresh thesis/data-to-visual/data-to-visual-nicos-branch/visualization_from_template.py:398: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


 17.76s  PASS 6/8 [--] chart=line        groups=1 annos=4  title="Monthly Sales Analysis (2021-2024)"
  Run 03/10  [3/30 | 0.8min]  ... 

/Users/srujanakadambari/Desktop/ffresh thesis/data-to-visual/data-to-visual-nicos-branch/visualization_from_template.py:398: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


 31.97s  PASS 8/8 [OK] chart=bar         groups=4 annos=4  title="Monthly Sales from 2021 to 2024"
  Run 04/10  [4/30 | 1.4min]  ... 

...

 23.35s  PASS 8/8 [OK] chart=line        groups=4 annos=8  title="Umsatz SD/CO Analysis for Teckentrup (20"
  Run 08/10  [28/30 | 9.0min]  ... 

/Users/srujanakadambari/Desktop/ffresh thesis/data-to-visual/data-to-visual-nicos-branch/visualization_from_template.py:398: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


 21.74s  PASS 8/8 [OK] chart=line        groups=4 annos=4  title="Annual Revenue Trends (2021-2024)"
  Run 09/10  [29/30 | 9.3min]  ... 

/Users/srujanakadambari/Desktop/ffresh thesis/data-to-visual/data-to-visual-nicos-branch/visualization_from_template.py:398: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


 21.07s  PASS 8/8 [OK] chart=line        groups=4 annos=6  title="Monthly Sales Trends and Annual Sales Co"
  Run 10/10  [30/30 | 9.7min]  ...  28.79s  PASS 6/8 [--] chart=line        groups=1 annos=3  title="Umsatz Analysis (2021-2024)"

  ALL 30 RUNS COMPLETE in 10.2 minutes
  Master CSV: master_run_results/master_pipeline_runs.csv
  Configs:    master_run_results/configs/  (30 JSONs)
  Renders:    master_run_results/renders/  (PNGs for successful renders)


## Cell 5 — Status summary

In [21]:
# ═══════════════════════════════════════════════════════════════════
# CELL 5 — Quick summary of what we have
# ═══════════════════════════════════════════════════════════════════
import pandas as pd

df = pd.read_csv(MASTER_CSV)
df["latency_s"] = pd.to_numeric(df["latency_s"], errors="coerce")
df["passed"] = df["passed"].astype(str).isin(["True","1","1.0"])
df["renderable"] = df["renderable"].astype(str).isin(["True","1","1.0"])

summary = df.groupby("scenario").agg(
    desc=("scenario_desc", "first"),
    n_runs=("run", "count"),
    n_pass=("passed", "sum"),
    n_render=("renderable", "sum"),
    lat_mean=("latency_s", "mean"),
    lat_min=("latency_s", "min"),
    lat_max=("latency_s", "max"),
    qual_mean=("quality_score", "mean"),
).reset_index()

print(f"\n{'='*100}\n  PIPELINE COMPLETION STATUS\n{'='*100}")
print(f"  {'Scenario':<6} {'Runs':>5} {'Pass':>5} {'Renders':>8} "
      f"{'Mean lat':>10} {'Min':>7} {'Max':>7} {'Qual':>6}")
print("-"*100)
for _, r in summary.iterrows():
    print(f"  {r['scenario']:<6} {int(r['n_runs']):>5} {int(r['n_pass']):>5} "
          f"{int(r['n_render']):>8} {r['lat_mean']:>7.1f}s   "
          f"{r['lat_min']:>5.1f}s {r['lat_max']:>5.1f}s  {r['qual_mean']:>4.2f}/8")
print("="*100)
print(f"\n  Total runs in master CSV: {len(df)}")
print(f"  Total renders created:    {df['renderable'].sum()}")
print(f"  Configs ready for Qwen judging: {len(df)} (run notebook 2 next)")



  PIPELINE COMPLETION STATUS
  Scenario  Runs  Pass  Renders   Mean lat     Min     Max   Qual
----------------------------------------------------------------------------------------------------
  S0         3     3        3    34.5s    33.3s  35.7s  8.00/8
  S3        10    10        8    20.3s    16.3s  24.9s  7.80/8
  S4        10    10        7    31.4s    20.6s  43.0s  7.50/8
  S4b       10    10        6    25.3s    15.4s  46.8s  7.30/8
  S5        10    10        8    38.0s    23.9s  54.0s  7.80/8
  S5b       10    10        8    24.4s    17.2s  55.8s  7.80/8
  S6        10    10        9    18.5s    12.9s  28.7s  7.90/8
  S7        10    10        4    37.6s    31.9s  46.0s  7.00/8
  S8        10    10        2    52.0s    30.3s  67.1s  6.50/8
  S9        10    10        8     7.6s     5.0s  13.6s  7.60/8
  SA1       10    10       10    51.7s    31.4s  63.2s  8.00/8
  SA2       10    10        3    39.1s    32.8s  51.8s  6.90/8
  SA3       10    10        7    22.9s    16.9s